# Task 1 — Word Occurrence Counting using Multithreading (Pthreads)
**6CS005 High Performance Computing**

This notebook:
1. Writes the C source file inline
2. Compiles it with GCC + pthreads
3. Creates a sample text file
4. Runs the programme and displays the output

## Step 1 — Write the C source file

In [ ]:
%%writefile word_count.c
/*
 * Task 1: Word Occurrence Counting using Multithreading (Pthreads)
 * 6CS005 - High Performance Computing
 */

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <pthread.h>

#define MAX_WORD_LEN  256
#define HASH_SIZE     65536
#define MAX_WORDS     200000

typedef struct Entry {
    char          word[MAX_WORD_LEN];
    long          count;
    struct Entry *next;
} Entry;

typedef struct {
    int    thread_id;
    char  *text;
    long   start;
    long   end;
    Entry *table[HASH_SIZE];
} ThreadArg;

static char  *g_text     = NULL;
static long   g_text_len = 0;
static int    g_nthreads = 1;
static Entry *g_table[HASH_SIZE];
static pthread_mutex_t g_mutex = PTHREAD_MUTEX_INITIALIZER;

static unsigned int fnv1a(const char *s) {
    unsigned int h = 2166136261u;
    while (*s) { h ^= (unsigned char)*s++; h *= 16777619u; }
    return h & (HASH_SIZE - 1);
}

static void local_insert(Entry **table, const char *word) {
    unsigned int idx = fnv1a(word);
    Entry *e = table[idx];
    while (e) { if (strcmp(e->word, word)==0){e->count++;return;} e=e->next; }
    e = (Entry*)malloc(sizeof(Entry));
    strncpy(e->word, word, MAX_WORD_LEN-1); e->word[MAX_WORD_LEN-1]='\0';
    e->count=1; e->next=table[idx]; table[idx]=e;
}

static void merge_into_global(Entry **lt) {
    pthread_mutex_lock(&g_mutex);
    for (int i=0;i<HASH_SIZE;i++) {
        Entry *e=lt[i];
        while(e) {
            unsigned int idx=fnv1a(e->word);
            Entry *ge=g_table[idx]; int found=0;
            while(ge){if(strcmp(ge->word,e->word)==0){ge->count+=e->count;found=1;break;}ge=ge->next;}
            if(!found){Entry *ne=(Entry*)malloc(sizeof(Entry));strcpy(ne->word,e->word);ne->count=e->count;ne->next=g_table[idx];g_table[idx]=ne;}
            e=e->next;
        }
    }
    pthread_mutex_unlock(&g_mutex);
}

static void free_table(Entry **table) {
    for(int i=0;i<HASH_SIZE;i++){Entry*e=table[i];while(e){Entry*nx=e->next;free(e);e=nx;}table[i]=NULL;}
}

static void *worker(void *arg) {
    ThreadArg *ta=(ThreadArg*)arg;
    char word[MAX_WORD_LEN]; int wlen=0;
    long s=ta->start;
    if(s>0 && !isspace((unsigned char)g_text[s-1]))
        while(s<ta->end && !isspace((unsigned char)g_text[s]))s++;
    long e=ta->end;
    if(e<g_text_len && !isspace((unsigned char)g_text[e-1]))
        while(e<g_text_len && !isspace((unsigned char)g_text[e]))e++;
    for(long i=s;i<e;i++){
        unsigned char c=(unsigned char)g_text[i];
        if(isalpha(c)||c=='\''){if(wlen<MAX_WORD_LEN-1)word[wlen++]=(char)tolower(c);}
        else{if(wlen>0){word[wlen]='\0';local_insert(ta->table,word);wlen=0;}}
    }
    if(wlen>0){word[wlen]='\0';local_insert(ta->table,word);}
    merge_into_global(ta->table);
    free_table(ta->table);
    return NULL;
}

typedef struct{char word[MAX_WORD_LEN];long count;}Pair;
static int cmp_desc(const void*a,const void*b){long d=((Pair*)b)->count-((Pair*)a)->count;return(d>0)-(d<0);}

int main(int argc,char*argv[]){
    if(argc<3){fprintf(stderr,"Usage: %s <filename> <num_threads>\n",argv[0]);return 1;}
    const char*filename=argv[1]; g_nthreads=atoi(argv[2]);
    if(g_nthreads<1){fprintf(stderr,"num_threads>=1\n");return 1;}
    FILE*fp=fopen(filename,"r"); if(!fp){perror("fopen");return 1;}
    fseek(fp,0,SEEK_END); g_text_len=ftell(fp); rewind(fp);
    g_text=(char*)malloc(g_text_len+1);
    fread(g_text,1,g_text_len,fp); g_text[g_text_len]='\0'; fclose(fp);
    printf("[INFO] File: %s | Size: %ld bytes | Threads: %d\n",filename,g_text_len,g_nthreads);
    memset(g_table,0,sizeof(g_table));
    pthread_t*tids=(pthread_t*)malloc(g_nthreads*sizeof(pthread_t));
    ThreadArg*args=(ThreadArg*)calloc(g_nthreads,sizeof(ThreadArg));
    long chunk=g_text_len/g_nthreads;
    for(int t=0;t<g_nthreads;t++){
        args[t].thread_id=t; args[t].text=g_text;
        args[t].start=t*chunk;
        args[t].end=(t==g_nthreads-1)?g_text_len:(t+1)*chunk;
        memset(args[t].table,0,sizeof(args[t].table));
        pthread_create(&tids[t],NULL,worker,&args[t]);
        printf("[INFO] Thread %d started (bytes %ld-%ld)\n",t,args[t].start,args[t].end);
    }
    for(int t=0;t<g_nthreads;t++) pthread_join(tids[t],NULL);
    printf("[INFO] All threads joined. Collecting results...\n");
    Pair*pairs=(Pair*)malloc(MAX_WORDS*sizeof(Pair));
    long total=0,unique=0;
    for(int i=0;i<HASH_SIZE;i++){
        Entry*e=g_table[i];
        while(e){if(unique<MAX_WORDS){strcpy(pairs[unique].word,e->word);pairs[unique].count=e->count;unique++;}total+=e->count;Entry*nx=e->next;free(e);e=nx;}
    }
    qsort(pairs,unique,sizeof(Pair),cmp_desc);
    FILE*out=fopen("result.txt","w");
    fprintf(out,"=== Word Frequency Report ===\nFile: %s\nThreads: %d\nTotal: %ld  Unique: %ld\n\n",filename,g_nthreads,total,unique);
    fprintf(out,"%-30s %10s\n%-30s %10s\n","Word","Frequency","------------------------------","----------");
    for(long i=0;i<unique;i++) fprintf(out,"%-30s %10ld\n",pairs[i].word,pairs[i].count);
    fclose(out);
    printf("[INFO] Results -> result.txt  (%ld unique, %ld total)\n",unique,total);
    free(g_text);free(tids);free(args);free(pairs);
    pthread_mutex_destroy(&g_mutex);
    return 0;
}

## Step 2 — Compile

In [ ]:
!gcc -O2 -o word_count word_count.c -lpthread && echo '✅ Compilation successful'

## Step 3 — Create sample text file

In [ ]:
sample_text = """
The quick brown fox jumps over the lazy dog.
The dog barked at the fox and the fox ran away quickly.
High performance computing requires careful parallel programming.
Parallel programming with threads can be challenging but rewarding.
The quick brown fox is a classic sentence used for testing purposes.
Multithreading allows multiple threads to run concurrently within a single process.
Each thread processes a distinct portion of data ensuring balanced workload distribution.
Synchronisation primitives such as mutexes protect shared data safely from race conditions.
The results are merged after all threads complete their assigned tasks successfully.
Word counting is a classic example of the map-reduce parallel programming model.
The fox and the dog are both animals that appear in many well known sentences.
Performance computing and parallel programming go hand in hand for speed improvements.
Threads share memory but must use synchronisation to avoid dangerous race conditions.
A mutex lock ensures that only one thread accesses shared data at any given time.
The final output contains each word alongside its frequency of occurrence in the file.
Sorting words alphabetically or by frequency provides very useful statistical insights.
High performance computing leverages parallelism to speed up complex computations greatly.
The quick brown fox jumps over the lazy dog is a well known pangram used for testing.
Pangrams contain every letter of the alphabet at least once making them useful for fonts.
Computing high performance tasks in parallel reduces overall execution time significantly.
The pthread library provides a standard interface for multithreaded programming in C.
Dynamic slicing of the workload ensures each thread receives a fair share of the data.
Mutex synchronisation prevents data corruption when multiple threads access shared memory.
The hash map data structure provides efficient word storage and lookup during counting.
After all threads complete their work the results are merged into a single output file.
"""

with open('sample.txt', 'w') as f:
    f.write(sample_text)

print(f'sample.txt created ({len(sample_text)} chars)')

## Step 4 — Run with different thread counts

In [ ]:
import subprocess, time

for n in [1, 2, 4, 8]:
    start = time.perf_counter()
    result = subprocess.run(['./word_count', 'sample.txt', str(n)],
                            capture_output=True, text=True)
    elapsed = time.perf_counter() - start
    print(f'--- {n} thread(s)  [{elapsed*1000:.1f} ms] ---')
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)

## Step 5 — Display result.txt

In [ ]:
with open('result.txt') as f:
    content = f.read()
print(content)

## Step 6 — Visualise top 20 words

In [ ]:
import matplotlib.pyplot as plt
import re

words, counts = [], []
for line in content.split('\n'):
    m = re.match(r'(\S+)\s+(\d+)', line.strip())
    if m:
        words.append(m.group(1))
        counts.append(int(m.group(2)))

top_n = 20
plt.figure(figsize=(14, 5))
plt.bar(words[:top_n], counts[:top_n], color='steelblue')
plt.title('Top 20 Most Frequent Words')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('word_freq.png', dpi=150)
plt.show()
print('Chart saved to word_freq.png')